# 虚拟细胞 · Demo

加载 4 模型集成，对新样本做扰动响应预测。`git clone` 后在项目根目录直接运行。

In [ ]:
import sys, numpy as np, pandas as pd, pickle, torch, importlib.util

ROOT = '.'
DATA = f'{ROOT}/data'
CODE = f'{ROOT}/code'
MODEL = f'{ROOT}/models'
sys.path.insert(0, CODE)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'设备: {DEV}')

In [ ]:
# ---------- 加载特征 ----------
meta = pd.read_pickle(f'{DATA}/meta.pkl')
feats = pickle.load(open(f'{DATA}/feats.pkl', 'rb'))
P = 4422
ctx_all = np.nan_to_num(feats['ctx_prior'].astype(np.float32), nan=0.0)
with open(f'{DATA}/prot_names.txt') as f:
    prot_names = f.read().splitlines()
print(f'特征加载完毕: {P} 蛋白, {len(meta)} 样本')

In [ ]:
# ---------- 加载 4 个模型 ----------
_s21 = importlib.util.spec_from_file_location('m21', f'{CODE}/04_model_v21.py')
_m21 = importlib.util.module_from_spec(_s21); _s21.loader.exec_module(_m21)
_s29 = importlib.util.spec_from_file_location('m29', f'{CODE}/04_model_v29.py')
_m29 = importlib.util.module_from_spec(_s29); _s29.loader.exec_module(_m29)

def load_model(path, cls, set_avg=False):
    m = cls(feats, P=P)
    m.load_state_dict(torch.load(f'{MODEL}/{path}', map_location=DEV, weights_only=True))
    if set_avg: m.set_strain_avg()
    return m.to(DEV).eval()

models = [
    (load_model('model_v21.pt', _m21.VCellModel), 'v21'),
    (load_model('model_v21_s43.pt', _m21.VCellModel), 'v21'),
    (load_model('model_v21_s44.pt', _m21.VCellModel), 'v21'),
    (load_model('model_v29_best.pt', _m29.VCellModel, set_avg=True), 'v29'),
]
print(f'{len(models)} 模型加载完毕: 3 x v2.1 + 1 x v2.9')

In [ ]:
# ---------- 推理 ----------
def ensemble_pred(idx):
    x = {
        'bio': [torch.from_numpy(feats['strain_id'][idx]), torch.from_numpy(feats['chem_id'][idx]),
                torch.from_numpy(feats['chem_hash'][idx]), torch.from_numpy(feats['medium_onehot'][idx]),
                torch.from_numpy(feats['temp_norm'][idx]), torch.from_numpy(feats['time_feat'][idx]),
                torch.from_numpy(feats['sm_id'][idx]), torch.from_numpy(feats['ct_id'][idx])],
        'ctx': [torch.from_numpy(feats['src_id'][idx]), torch.from_numpy(feats['ins_id'][idx]),
                torch.from_numpy(feats['plt_id'][idx])],
        'seen': [torch.from_numpy(feats['chem_seen'][idx]), torch.from_numpy(feats['strain_seen'][idx])],
        'ctx_prior': torch.from_numpy(ctx_all[idx]),
    }
    preds = []
    with torch.no_grad():
        for m, tag in models:
            xg = {k: (v.to(DEV) if k == 'ctx_prior' else [t.to(DEV) for t in v]) for k, v in x.items()}
            preds.append(m(xg).cpu().numpy())
    return np.mean(preds, axis=0)

In [ ]:
# ---------- 预测 val 集各场景前 2 个样本 ----------
for scene in ['val_chem_only', 'val_strain_only', 'val_both', 'val_time']:
    idx = np.where(meta['split_final'].eq(scene).values & meta['role'].eq('treatment').values)[0][:2]
    if len(idx) == 0: continue
    pred = ensemble_pred(idx)
    print(f'\n=== {scene} ===')
    for i, sid in enumerate(idx):
        r = meta.iloc[sid]
        print(f'  {r.name}: {r["Strains"]} | {r["perturbation_no_concentration"]} | '
              f'{r["Medium"]} | {r["Temperature"]} | {r["pert_time"]}')
        top3 = np.argsort(pred[i])[-3:][::-1]
        for pi in top3:
            print(f'      {prot_names[pi]:<10s} = {pred[i][pi]:.2f}')

In [ ]:
# ============================================================
# 预计算结果（来自完整训练日志和 _compare.py）
# ============================================================
print('\n' + '='*55)
print('val 集完整评估（4 模型集成）')
print('='*55)
print(f'{"场景":<20}{"样本":>6}{"蛋白R2中位":>11}{"FC PCC":>9}')
print('-'*50)
for scene, n, r2, fc in [
    ('val_chem_only', 1065, 0.879, 0.457),
    ('val_strain_only', 1333, 0.667, 0.350),
    ('val_both', 269, 0.760, 0.237),
    ('val_time', 139, 0.832, 0.610),
]:
    print(f'{scene:<20}{n:>6}{r2:>11.3f}{fc:>9.3f}')

In [ ]:
# ---------- 模型概览 ----------
print('\n' + '='*55)
print('v2.9 架构')
print('='*55)
m = models[3][0]
total = sum(pp.numel() for pp in m.parameters())
print(f'总参数: {total:,} ({total/1e6:.1f}M)')
t = sum(pp.numel() for pp in m.enc.parameters())/1e6
print(f'  混合编码器(B/S): {t:.2f}M')
t = sum(pp.numel() for pp in m.enc_C.parameters())/1e6
print(f'  C分支(化合物侧):  {t:.2f}M')
t = sum(pp.numel() for pp in m.enc_T.parameters())/1e6
print(f'  T分支(菌株侧):    {t:.2f}M')
t = sum(pp.numel() for pp in m.interact_mlp.parameters())/1e6
print(f'  交互项(菌株x药):  {t:.2f}M')
t = sum(pp.numel() for pp in m.proj.parameters())/1e3
print(f'  投影层:           {t:.1f}K')
t = sum(pp.numel() for pp in m.calib.parameters())/1e6
print(f'  校准分支:         {t:.2f}M')
gc = torch.sigmoid(m.gate_c).item()
gs = torch.sigmoid(m.gate_s).item()
gi = torch.sigmoid(m.gate_i).item()
print(f'门控: g_c={gc:.3f}  g_s={gs:.3f}  g_i={gi:.3f}')

## 复现

完整训练流程见 README.md：

```bash
python code/01_data_prep.py      # 预处理
python code/02_features.py       # 特征工程  
python code/05b_train_v21.py 42  # 训练 v2.1 (重复 43/44)
python code/05q_train_v29.py 42  # 训练 v2.9
python code/07e_submit_v29.py    # 生成提交文件
```

Python 3.10+, PyTorch 2.12.0+。